# COVID Parcel Business Analysis - Final Project
## Group 7
### Team Members:
- **Abeez Maredia**
- **Urvish Nayak**
- **Ayman Karovadiya**

### Project Overview:
This project analyzes the impact of COVID-19 on a parcel delivery business. We examine 
delivery volumes, revenue trends, regional performance, and operational efficiency 
during the pandemic period. Our goal is to provide executive-level insights and 
actionable recommendations.

### Date: April 2026

# 1. Importing Necessary Libraries

In [5]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# 2. Loading Data

In [6]:
df = pd.read_csv("COVID_Parcel_Business.csv")
print("Columns in dataset:", df.columns)

Columns in dataset: Index(['FakeCustomerID', 'THE_YEAR', 'THE_WEEK', 'VOLUME'], dtype='str')


# 3. Standarizing Column Names

In [7]:
year_col = None
cust_col = None
week_col = None
parcel_col = None

for col in df.columns:
    col_lower = col.lower().strip()

    if 'year' in col_lower:
        year_col = col
    elif 'cust' in col_lower or 'id' in col_lower:
        cust_col = col
    elif 'week' in col_lower:
        week_col = col
    elif 'volume' in col_lower or 'parcel' in col_lower or 'vol' in col_lower:
        parcel_col = col

# Checking if all columns found
if year_col and cust_col and week_col and parcel_col:
    print("Columns detected successfully!")
else:
    print("Error: Could not detect required columns")

Columns detected successfully!


# 4. Defining Business Rules

In [8]:
BASE_COST = 22.0

SEGMENT_DEFS = {
    'Enterprise': {'min': 500001, 'max': float('inf'), 'discount': 0.22},
    'Large': {'min': 200001, 'max': 500000, 'discount': 0.17},
    'Medium': {'min': 10001, 'max': 200000, 'discount': 0.10},
    'Small': {'min': 1000, 'max': 10000, 'discount': 0.04}
}

# 5. weekly trend analysis

In [ ]:
weekly = df.groupby([year_col, week_col])[parcel_col].sum().reset_index()

plt.figure()
for year in weekly[year_col].unique():
    temp = weekly[weekly[year_col] == year]
    plt.plot(temp[week_col], temp[parcel_col], label=f"Year {year}")

plt.legend()
plt.title("Weekly Parcel Volume (2019 vs 2020)")
plt.xlabel("Week")
plt.ylabel("Parcels")
plt.show()

# 6. pre-covid vs covid split

In [ ]:
pre_covid_2020 = df[(df[year_col] == 2020) & (df[week_col] <= 15)]
covid_2020 = df[(df[year_col] == 2020) & (df[week_col] > 15)]

weekly_pre = pre_covid_2020.groupby(week_col)[parcel_col].sum()
weekly_covid = covid_2020.groupby(week_col)[parcel_col].sum()

plt.figure()
plt.plot(weekly_pre.index, weekly_pre.values, label='Pre-COVID')
plt.plot(weekly_covid.index, weekly_covid.values, label='COVID')

plt.legend()
plt.title("Weekly Parcel Trend (2020)")
plt.xlabel("Week")
plt.ylabel("Parcels")
plt.show()

# 7. ISGR calculation

In [ ]:
pre_covid_2020 = df[(df[year_col] == 2020) & (df[week_col] <= 15)]
covid_2020 = df[(df[year_col] == 2020) & (df[week_col] > 15)]

pre_2019 = df[(df[year_col] == 2019) & (df[week_col] <= 15)]


vol_2020 = pre_covid_2020[parcel_col].sum()
vol_2019 = pre_2019[parcel_col].sum()

isgr = ((vol_2020 - vol_2019) / vol_2019) * 100
print("Industry Standard Growth Rate (ISGR):", round(isgr,2), "%")

# 8. customer level analysis

In [ ]:
baseline = pre_2019.groupby(cust_col)[parcel_col].sum()
covid = covid_2020.groupby(cust_col)[parcel_col].sum()

final_df = pd.DataFrame({
    'Vol_Baseline': baseline,
    'Vol_COVID': covid
}).fillna(0)

final_df['Growth_Rate'] = np.where(
    final_df['Vol_Baseline'] == 0,
    np.nan,
    (final_df['Vol_COVID'] - final_df['Vol_Baseline']) / final_df['Vol_Baseline'] * 100
)

print(final_df.head())